# Building the kid demo gallery

This notebook documents how the gallery for the [kids' Space](https://huggingface.co/spaces/quidditch/frozen-ai-kids) ("How does the computer know?") was built.

The key idea: the "which part did the computer look at?" answer for each image is **not** picked by hand. It is read off the model's own **Grad-CAM** attention map. The model says *who* it is; Grad-CAM shows *where it looked*; that hot region becomes the pre-labeled correct answer in the Space's `app.py`. Inference is deterministic, so a given image always produces the same hot region, which is why the labels can be hardcoded once.

Two steps are documented here:
1. Grad-CAM each candidate image and read off the hottest region (the label).
2. Pad each image to a square thumbnail for a clean, uniform gallery grid.

In [ ]:
!pip install -q fastai
from fastai.vision.all import *
import os, urllib.request, matplotlib.pyplot as plt
from PIL import Image

REPO = "https://raw.githubusercontent.com/megano/deep-learning-image-classifier/master"
if not os.path.exists("export.pkl"):
    urllib.request.urlretrieve(f"{REPO}/export.pkl", "export.pkl")
learn = load_learner("export.pkl")
print("classes:", list(learn.dls.vocab))

## 1. Grad-CAM: where did the model look?

Hooks the ResNet-18 body's final feature map and weights it by the gradient of the predicted class. Bright spots are the pixels that pushed the prediction.

In [ ]:
class Hook:
    def __init__(s, m): s.h = m.register_forward_hook(s.f)
    def f(s, m, i, o): s.stored = o.detach().clone()
    def __enter__(s, *a): return s
    def __exit__(s, *a): s.h.remove()

class HookBwd:
    def __init__(s, m): s.h = m.register_full_backward_hook(s.f)
    def f(s, m, gi, go): s.stored = go[0].detach().clone()
    def __enter__(s, *a): return s
    def __exit__(s, *a): s.h.remove()

def heatmap(fp, ax):
    learn.model.zero_grad()
    x, = first(learn.dls.test_dl([PILImage.create(fp)]))
    with HookBwd(learn.model[0]) as hb, Hook(learn.model[0]) as h:
        out = learn.model.eval()(x); act = h.stored[0]
        idx = int(out.argmax(1).item()); out[0, idx].backward(); grad = hb.stored[0]
    cam = (grad.mean([1, 2], keepdim=True) * act).sum(0).relu()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    ax.imshow(PILImage.create(fp).resize((224, 224)))
    ax.imshow(cam.detach().cpu().numpy(), alpha=0.5, extent=(0, 224, 224, 0),
              cmap="magma", interpolation="bilinear")
    ax.set_title(f"{os.path.basename(fp)} -> {learn.dls.vocab[idx]}", fontsize=9)
    ax.axis("off")

## 2. Read the hot region for each gallery image

For each character we tested several candidates and kept the ones with a clear, namable hot spot, and enough variety that the answer is not always "the face." Running the final nine shows where each one's heat lands.

In [ ]:
FILES = [
    "olaf_009.png", "olaf_028.png", "olaf_004.png",
    "elsa_026.jpg", "elsa_084.jpg", "elsa_107.jpg",
    "sven_004.jpg", "sven_005.jpg", "sven_008.jpg",
]
os.makedirs("gallery", exist_ok=True)
for f in FILES:
    urllib.request.urlretrieve(f"{REPO}/hf-kid-space/examples/{f}", f"gallery/{f}")

fig, axs = plt.subplots(3, 3, figsize=(11, 11))
for f, ax in zip(FILES, axs.ravel()): heatmap(f"gallery/{f}", ax)
plt.tight_layout()

The hottest region for each image becomes its answer in `app.py`'s `CORRECT_PART`:

| Image | Looked at most |
|---|---|
| olaf_009 | Face |
| olaf_028 | Carrot nose |
| olaf_004 | Face (heat spreads over the body but peaks on the face) |
| elsa_026 | Hands |
| elsa_084 | Dress / body |
| elsa_107 | Hair / braid |
| sven_004, sven_005, sven_008 | Fuzzy face & mane |

The model gravitates to faces for the human (Elsa), so getting a clean "braid" or "hands" example took testing several images. The "Antlers" option for Sven is a deliberate distractor: the model actually keys on the fuzzy mane, not the antlers.

## 3. Square thumbnails for a uniform grid

The source images have mixed aspect ratios, which makes an uneven gallery. We pad each to a square (white background) for display only. The model still runs on the full-resolution originals, so predictions and heatmaps are unaffected.

In [ ]:
def make_square_thumb(src, dst, size=400):
    im = Image.open(src).convert("RGB")
    w, h = im.size; s = max(w, h)
    canvas = Image.new("RGB", (s, s), (255, 255, 255))
    canvas.paste(im, ((s - w) // 2, (s - h) // 2))
    canvas.resize((size, size), Image.LANCZOS).save(dst)

os.makedirs("thumbs", exist_ok=True)
for f in FILES:
    make_square_thumb(f"gallery/{f}", f"thumbs/{f}")
print("wrote", len(FILES), "square thumbnails")

fig, axs = plt.subplots(3, 3, figsize=(9, 9))
for f, ax in zip(FILES, axs.ravel()):
    ax.imshow(Image.open(f"thumbs/{f}")); ax.axis("off")
plt.tight_layout()

That is the full pipeline: Grad-CAM gives the honest "where it looked" label, and the square padding gives the clean grid. Both feed directly into the kids' Space `app.py`.